# 1. Setup & Data Exploration

### 0.0 SET UP

In [ ]:

-- 1. Create database 


IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = 'EDA')
BEGIN
    CREATE DATABASE EDA;
END
GO

USE EDA;
GO


-- 2. Crate table (drop if exists)


IF OBJECT_ID('dbo.HR_Employee_Attrition', 'U') IS NOT NULL
    DROP TABLE dbo.HR_Employee_Attrition;
GO


-- 3. Create a table according to the dataset structure


CREATE TABLE dbo.HR_Employee_Attrition (
    Age                      INT,
    Attrition                VARCHAR(5),
    BusinessTravel           VARCHAR(50),
    DailyRate                INT,
    Department               VARCHAR(100),
    DistanceFromHome         INT,
    Education                INT,
    EducationField           VARCHAR(100),
    EmployeeCount            INT,
    EmployeeNumber           INT,
    EnvironmentSatisfaction  INT,
    Gender                   VARCHAR(10),
    HourlyRate               INT,
    JobInvolvement           INT,
    JobLevel                 INT,
    JobRole                  VARCHAR(100),
    JobSatisfaction          INT,
    MaritalStatus            VARCHAR(20),
    MonthlyIncome            INT,
    MonthlyRate              INT,
    NumCompaniesWorked       INT,
    Over18                   VARCHAR(5),
    OverTime                 VARCHAR(5),
    PercentSalaryHike        INT,
    PerformanceRating        INT,
    RelationshipSatisfaction INT,
    StandardHours            INT,
    StockOptionLevel         INT,
    TotalWorkingYears        INT,
    TrainingTimesLastYear    INT,
    WorkLifeBalance          INT,
    YearsAtCompany           INT,
    YearsInCurrentRole       INT,
    YearsSinceLastPromotion  INT,
    YearsWithCurrManager     INT
);
GO


-- 4. Import data


BULK INSERT dbo.HR_Employee_Attrition
FROM 'D:\My Project Datasets\EDA\HR-Employee-Attrition.csv'
WITH (
    FORMAT          = 'CSV',
    FIRSTROW        = 2,
    FIELDTERMINATOR = ',',
    ROWTERMINATOR   = '\n',
    CODEPAGE        = '65001'
);
GO

<br><br>

### 1.1 Check total rows & total columns

##### - Total rows

In [1]:
SELECT COUNT(*) AS total_rows FROM dbo.HR_Employee_Attrition;

(1 row affected)

total_rows
----------
1470      
(1 row)

<b>

##### - Total columns


In [2]:

SELECT COUNT(*) AS total_columns
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'HR_Employee_Attrition';

(1 row affected)

total_columns
-------------
35           
(1 row)

<b><b>

The dataset contains **1,470 employee records** across **35 attributes**, covering demographics, job details, compensation, satisfaction scores, and work conditions. This is a mid-sized dataset — large enough to detect meaningful patterns, but small enough that subgroup analyses (e.g., per department) may have limited sample sizes in smaller segments.

<b>

### 1.2 Check the attrition distribution (what percentage resigns vs. stays)

In [3]:
SELECT 
    Attrition,
    COUNT(*) AS total,
    CAST(ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS DECIMAL(5,2)) AS percentage
FROM dbo.HR_Employee_Attrition
GROUP BY Attrition;

(2 rows affected)

Attrition | total | percentage
----------+-------+-----------
Yes       | 237   | 16.12     
No        | 1233  | 83.88     
(2 rows)

<b>

Of the total 1,470 employees, 237 (16.12%) have left the company, while 1,233 (83.88%) remain. This 16% figure is quite high — generally, a healthy attrition rate is below 10% per year.

<br><b>

### 1.3 Summary statistics: average age, income, years at company

In [4]:
SELECT 
    CAST(AVG(Age * 1.0)            AS DECIMAL(5,2)) AS avg_age,
    CAST(AVG(MonthlyIncome * 1.0)  AS DECIMAL(10,2)) AS avg_monthly_income,
    CAST(AVG(YearsAtCompany * 1.0) AS DECIMAL(5,2)) AS avg_years_at_company
FROM dbo.HR_Employee_Attrition;


(1 row affected)

avg_age | avg_monthly_income | avg_years_at_company
--------+--------------------+---------------------
36.92   | 6502.93            | 7.01                
(1 row)

The average employee is approximately **37 years old**, earns **$6,503/month**, and has been with the company for about **7 years**. These figures establish a baseline — any subgroup that deviates significantly from these averages (e.g., younger employees leaving earlier, lower-income employees resigning more) will be worth investigating in the analysis ahead.

<br><b>

### 1.4 Distribution of employees per department, job role, education field

##### - Per Department

In [5]:
SELECT Department, COUNT(*) AS total
FROM dbo.HR_Employee_Attrition
GROUP BY Department
ORDER BY total DESC;

(3 rows affected)

Department             | total
-----------------------+------
Research & Development | 961  
Sales                  | 446  
Human Resources        | 63   
(3 rows)

<b>

The Research & Development department dominates with 961 employees (65.37%), followed by Sales (30.34%), and Human Resources (4.29%). This indicates a research- and sales-driven company.

<b>

##### - Per Job Role

In [6]:
SELECT JobRole, COUNT(*) AS total
FROM dbo.HR_Employee_Attrition
GROUP BY JobRole
ORDER BY total DESC;

(9 rows affected)

JobRole                   | total
--------------------------+------
Sales Executive           | 326  
Research Scientist        | 292  
Laboratory Technician     | 259  
Manufacturing Director    | 145  
Healthcare Representative | 131  
Manager                   | 102  
Sales Representative      | 83   
Research Director         | 80   
Human Resources           | 52   
(9 rows)

<b>

The three positions with the largest number of employees are:

- Sales Executive (326 people, 22.18%)
- Research Scientist (292 people, 19.86%)
- Laboratory Technician (259 people, 17.62%)

These three positions account for more than half of the company's total employees.

<br>

##### - Per Education Field

In [7]:

SELECT EducationField, COUNT(*) AS total
FROM dbo.HR_Employee_Attrition
GROUP BY EducationField
ORDER BY total DESC;

(6 rows affected)

EducationField   | total
-----------------+------
Life Sciences    | 606  
Medical          | 464  
Marketing        | 159  
Technical Degree | 132  
Other            | 82   
Human Resources  | 27   
(6 rows)

<b>

**Observable pattern:**

- Sales Representative positions have the highest turnover risk — nearly 4 out of 10 employees in this position ultimately resign. This is likely related to target pressure, commission-based compensation, and limited career paths.
- Laboratory Technicians and HR are also in the red zone (>20%), requiring special attention.
- The higher the position level (Manager, Research Director), the lower the turnover rate — suggesting that seniority and position influence retention.

<br>

#### 💡 Section 1 Insight: Data Exploration Summary

This company is **R&D-heavy** (65% of workforce) with a **16.12% overall attrition rate** — well above the healthy benchmark of ~10%. The workforce is mid-career (avg age 37, avg tenure 7 years). The top 3 roles (Sales Executive, Research Scientist, Lab Technician) make up nearly 60% of all employees, so attrition in these roles will have an outsized business impact. The next sections will investigate *why* employees are leaving.

---

<br>

# 2. Business Analysis 

<br>

### 2.1 Attrition by Demographics

##### - Attrition Rate per Department

In [8]:
SELECT
    Department,
    COUNT(*)                                                                   AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                         AS Resigned,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)                         AS Retained,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                           AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY Department
ORDER BY AttritionRate DESC;

(3 rows affected)

Department             | TotalEmployee | Resigned | Retained | AttritionRate
-----------------------+---------------+----------+----------+--------------
Sales                  | 446           | 92       | 354      | 20.63        
Human Resources        | 63            | 12       | 51       | 19.05        
Research & Development | 961           | 133      | 828      | 13.84        
(3 rows)

<br>

**Sales** has the highest attrition rate, followed by **Human Resources** and **R&D**. While R&D has the largest absolute number of resignations (due to its size), Sales employees are proportionally the most likely to leave. HR's small team size means even a few departures create a significant percentage impact.

<b><b>

##### - Attrition rate per age group

In [9]:
SELECT
    CASE
        WHEN Age < 25              THEN '< 25'
        WHEN Age BETWEEN 25 AND 34 THEN '25 - 34'
        WHEN Age BETWEEN 35 AND 44 THEN '35 - 44'
        WHEN Age BETWEEN 45 AND 54 THEN '45 - 54'
        ELSE '55+'
    END                                                                         AS AgeGroup,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)                          AS Retained,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY
    CASE
        WHEN Age < 25              THEN '< 25'
        WHEN Age BETWEEN 25 AND 34 THEN '25 - 34'
        WHEN Age BETWEEN 35 AND 44 THEN '35 - 44'
        WHEN Age BETWEEN 45 AND 54 THEN '45 - 54'
        ELSE '55+'
    END
ORDER BY AttritionRate DESC;


(5 rows affected)

AgeGroup | TotalEmployee | Resigned | Retained | AttritionRate
---------+---------------+----------+----------+--------------
< 25     | 97            | 38       | 59       | 39.18        
25 - 34  | 554           | 112      | 442      | 20.22        
55+      | 69            | 11       | 58       | 15.94        
45 - 54  | 245           | 25       | 220      | 10.20        
35 - 44  | 505           | 51       | 454      | 10.10        
(5 rows)

<br>

The **under-25** and **25-34** age groups show the highest attrition rates. This is consistent with industry norms — younger employees tend to job-hop more frequently, especially early in their careers when they're still exploring options. The **45+** group has the lowest attrition, suggesting that employees who reach mid-career at this company tend to stay.

<b><b>

##### - Attrition by gender & marital status — is there a pattern ?

In [10]:
-- Attrition by Gender


SELECT
    Gender,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)                          AS Retained,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY Gender
ORDER BY AttritionRate DESC;


(2 rows affected)

Gender | TotalEmployee | Resigned | Retained | AttritionRate
-------+---------------+----------+----------+--------------
Male   | 882           | 150      | 732      | 17.01        
Female | 588           | 87       | 501      | 14.80        
(2 rows)

<br>

Gender alone shows a moderate difference in attrition rates, with **male employees** having slightly higher turnover. However, gender in isolation may not be the full story — the cross-analysis below reveals that **marital status** is a much stronger predictor.

<b><b>

In [11]:

-- Attrition by Marital Status


SELECT
    MaritalStatus,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)                          AS Retained,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY MaritalStatus
ORDER BY AttritionRate DESC;


(3 rows affected)

MaritalStatus | TotalEmployee | Resigned | Retained | AttritionRate
--------------+---------------+----------+----------+--------------
Single        | 470           | 120      | 350      | 25.53        
Married       | 673           | 84       | 589      | 12.48        
Divorced      | 327           | 33       | 294      | 10.09        
(3 rows)

<br>

**Single employees** have the highest attrition rate by a significant margin, regardless of gender. This aligns with the idea that single employees have fewer location/stability commitments and are more willing to switch jobs. **Divorced** employees, interestingly, have the lowest attrition — possibly because they prioritize job stability.

<b><b>

In [12]:

-- Attrition by Gender x Marital Status (Cross Analysis)


SELECT
    Gender,
    MaritalStatus,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY Gender, MaritalStatus
ORDER BY AttritionRate DESC;


(6 rows affected)

Gender | MaritalStatus | TotalEmployee | Resigned | AttritionRate
-------+---------------+---------------+----------+--------------
Male   | Single        | 271           | 73       | 26.94        
Female | Single        | 199           | 47       | 23.62        
Male   | Married       | 401           | 53       | 13.22        
Male   | Divorced      | 210           | 24       | 11.43        
Female | Married       | 272           | 31       | 11.40        
Female | Divorced      | 117           | 9        | 7.69         
(6 rows)

<br><br>

The cross-analysis confirms: **marital status matters more than gender**. Single males (26.94%) and single females (23.62%) both have attrition rates roughly **2-3x higher** than their married or divorced counterparts. The gap between genders within the same marital status is relatively small (3-4 percentage points), while the gap between marital statuses within the same gender is massive (up to 15+ points).

#### 💡 Section 2.1 Insight: Demographics

Attrition is driven more by **life stage** than demographics. Young, single employees are the highest-risk group — particularly single males under 34. Department-wise, Sales consistently loses the most talent. HR teams should consider targeted retention efforts for early-career, single employees in customer-facing roles.

<b><b><b>

### 2.2 Attrition by Compensation & Satisfaction

##### - Average monthly income: those who resign vs. those who stay — what is the gap?

In [13]:
-- Average Monthly Income: Resigned vs Retained


SELECT
    Attrition,
    COUNT(*)                                    AS TotalEmployee,
    CAST(AVG(MonthlyIncome) AS DECIMAL(10,2))   AS AvgMonthlyIncome,
    MIN(MonthlyIncome)                          AS MinIncome,
    MAX(MonthlyIncome)                          AS MaxIncome
FROM dbo.HR_Employee_Attrition
GROUP BY Attrition
ORDER BY Attrition DESC;

(2 rows affected)

Attrition | TotalEmployee | AvgMonthlyIncome | MinIncome | MaxIncome
----------+---------------+------------------+-----------+----------
Yes       | 237           | 4787.00          | 1009      | 19859    
No        | 1233          | 6832.00          | 1051      | 19999    
(2 rows)

<br>

Employees who resigned earned **$4,787/month** on average — roughly **30% less** than those who stayed ($6,832). The minimum incomes are similar (~$1,000), but the average gap suggests that lower-paid employees are significantly more likely to leave. However, correlation is not causation — lower income could also reflect shorter tenure or junior roles.

<b><b>

In [14]:
-- Income Gap


SELECT
    CAST(AVG(CASE WHEN Attrition = 'No'  THEN CAST(MonthlyIncome AS FLOAT) END) AS DECIMAL(10,2)) AS AvgIncome_Stay,
    CAST(AVG(CASE WHEN Attrition = 'Yes' THEN CAST(MonthlyIncome AS FLOAT) END) AS DECIMAL(10,2)) AS AvgIncome_Resign,
    CAST(AVG(CASE WHEN Attrition = 'No'  THEN CAST(MonthlyIncome AS FLOAT) END)
       - AVG(CASE WHEN Attrition = 'Yes' THEN CAST(MonthlyIncome AS FLOAT) END)
         AS DECIMAL(10,2))                                                        AS IncomeGap,
    CAST((AVG(CASE WHEN Attrition = 'No'  THEN CAST(MonthlyIncome AS FLOAT) END)
        - AVG(CASE WHEN Attrition = 'Yes' THEN CAST(MonthlyIncome AS FLOAT) END))
        * 100.0
        / AVG(CASE WHEN Attrition = 'No'  THEN CAST(MonthlyIncome AS FLOAT) END)
         AS DECIMAL(5,2))                                                         AS GapPercentage
FROM dbo.HR_Employee_Attrition;

(1 row affected)

AvgIncome_Stay | AvgIncome_Resign | IncomeGap | GapPercentage
---------------+------------------+-----------+--------------
6832.74        | 4787.09          | 2045.65   | 29.94        
(1 row)

<br>

The **$2,046 income gap** (29.94%) is substantial. For every dollar a retained employee earns, a departing employee earned only about **70 cents**. This gap alone doesn't prove that low pay causes attrition, but it's a strong signal that compensation is a major factor.

<b><b>

In [15]:
-- Breakdown per Department: income gap resign vs stay


SELECT
    Department,
    CAST(AVG(CASE WHEN Attrition = 'No'  THEN CAST(MonthlyIncome AS FLOAT) END) AS DECIMAL(10,2)) AS AvgIncome_Stay,
    CAST(AVG(CASE WHEN Attrition = 'Yes' THEN CAST(MonthlyIncome AS FLOAT) END) AS DECIMAL(10,2)) AS AvgIncome_Resign,
    CAST(AVG(CASE WHEN Attrition = 'No'  THEN CAST(MonthlyIncome AS FLOAT) END)
       - AVG(CASE WHEN Attrition = 'Yes' THEN CAST(MonthlyIncome AS FLOAT) END)
         AS DECIMAL(10,2))                                                        AS IncomeGap
FROM dbo.HR_Employee_Attrition
GROUP BY Department
ORDER BY IncomeGap DESC;

(3 rows affected)

Department             | AvgIncome_Stay | AvgIncome_Resign | IncomeGap
-----------------------+----------------+------------------+----------
Human Resources        | 7345.98        | 3715.75          | 3630.23  
Research & Development | 6630.33        | 4108.08          | 2522.25  
Sales                  | 7232.24        | 5908.46          | 1323.78  
(3 rows)

<br>

**Human Resources** has the largest income gap ($3,630) between those who stay and those who leave — meaning HR employees who resign were earning dramatically less than their staying colleagues. Interestingly, **Sales** has the smallest gap ($1,324), suggesting that Sales attrition may be driven more by work conditions (overtime, travel) than compensation alone.

<b><b><b>

##### - Attrition rate per salary band

In [16]:
-- Attrition Rate per Salary Quartile (NTILE)


WITH SalaryQuartile AS (
    SELECT
        EmployeeNumber,
        Attrition,
        MonthlyIncome,
        NTILE(4) OVER (ORDER BY MonthlyIncome)  AS Quartile
    FROM dbo.HR_Employee_Attrition
)
SELECT
    Quartile,
    CASE Quartile
        WHEN 1 THEN 'Q1 - Low'
        WHEN 2 THEN 'Q2 - Mid-Low'
        WHEN 3 THEN 'Q3 - Mid-High'
        WHEN 4 THEN 'Q4 - High'
    END                                                                         AS SalaryBand,
    COUNT(*)                                                                    AS TotalEmployee,
    MIN(MonthlyIncome)                                                          AS MinIncome,
    MAX(MonthlyIncome)                                                          AS MaxIncome,
    CAST(AVG(CAST(MonthlyIncome AS FLOAT)) AS DECIMAL(10,2))                    AS AvgIncome,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM SalaryQuartile
GROUP BY Quartile
ORDER BY Quartile;


(4 rows affected)

Quartile | SalaryBand    | TotalEmployee | MinIncome | MaxIncome | AvgIncome | Resigned | AttritionRate
---------+---------------+---------------+-----------+-----------+-----------+----------+--------------
1        | Q1 - Low      | 368           | 1009      | 2911      | 2352.62   | 108      | 29.35        
2        | Q2 - Mid-Low  | 368           | 2911      | 4930      | 3963.87   | 52       | 14.13        
3        | Q3 - Mid-High | 367           | 4936      | 8380      | 6191.14   | 39       | 10.63        
4        | Q4 - High     | 367           | 8381      | 19999     | 13522.33  | 38       | 10.35        
(4 rows)

<br>

The salary quartile analysis confirms a **clear linear relationship**: the lower the salary bracket, the higher the attrition rate. Employees in the **bottom 25% salary bracket** resign at roughly **2-3x the rate** of those in the top quartile. This is one of the strongest predictors of attrition in the dataset.

<b><b><b>

##### - Job satisfaction vs attrition — do those with low satisfaction resign more?

In [17]:
-- Attrition Rate per Job Satisfaction Level


SELECT
    JobSatisfaction,
    CASE JobSatisfaction
        WHEN 1 THEN '1 - Low'
        WHEN 2 THEN '2 - Medium'
        WHEN 3 THEN '3 - High'
        WHEN 4 THEN '4 - Very High'
    END                                                                         AS SatisfactionLabel,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)                          AS Retained,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY JobSatisfaction
ORDER BY JobSatisfaction;

(4 rows affected)

JobSatisfaction | SatisfactionLabel | TotalEmployee | Resigned | Retained | AttritionRate
----------------+-------------------+---------------+----------+----------+--------------
1               | 1 - Low           | 289           | 66       | 223      | 22.84        
2               | 2 - Medium        | 280           | 46       | 234      | 16.43        
3               | 3 - High          | 442           | 73       | 369      | 16.52        
4               | 4 - Very High     | 459           | 52       | 407      | 11.33        
(4 rows)

<b>

Per level satisfaction (1–4) — see if there is a linear trend: the lower the satisfaction, the higher the attrition

<br><b>

In [18]:
-- Cross: Job Satisfaction x Department


SELECT
    Department,
    JobSatisfaction,
    CASE JobSatisfaction
        WHEN 1 THEN '1 - Low'
        WHEN 2 THEN '2 - Medium'
        WHEN 3 THEN '3 - High'
        WHEN 4 THEN '4 - Very High'
    END                                                                         AS SatisfactionLabel,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY Department, JobSatisfaction
ORDER BY Department, JobSatisfaction;


(12 rows affected)

Department             | JobSatisfaction | SatisfactionLabel | TotalEmployee | Resigned | AttritionRate
-----------------------+-----------------+-------------------+---------------+----------+--------------
Human Resources        | 1               | 1 - Low           | 11            | 5        | 45.45        
Human Resources        | 2               | 2 - Medium        | 20            | 2        | 10.00        
Human Resources        | 3               | 3 - High          | 15            | 3        | 20.00        
Human Resources        | 4               | 4 - Very High     | 17            | 2        | 11.76        
Research & Development | 1               | 1 - Low           | 192           | 38       | 19.79        
Research & Development | 2               | 2 - Medium        | 174           | 24       | 13.79        
Research & Development | 3               | 3 - High          | 300           | 43       | 14.33        
Research & Development | 4               | 4 - Very High     | 2

<br><b>

In [ ]:
-- Average Job Satisfaction: Resign vs Stay


SELECT
    Attrition,
    CAST(AVG(CAST(JobSatisfaction AS FLOAT)) AS DECIMAL(4,2))    AS AvgJobSatisfaction
FROM dbo.HR_Employee_Attrition
GROUP BY Attrition;

(2 rows affected)

Attrition | AvgJobSatisfaction
----------+-------------------
Yes       | 2.47              
No        | 2.78              
(2 rows)

<br><br>

Employees who resigned had an average job satisfaction of **2.47** vs **2.78** for those who stayed (on a 1-4 scale). While the gap exists, it's smaller than the income gap — suggesting that **compensation has a stronger pull on attrition than satisfaction alone**. However, when combined with other factors (overtime, low pay), low satisfaction may be the tipping point.

#### 💡 Section 2.2 Insight: Compensation & Satisfaction

Compensation is a **dominant attrition driver**. Employees who leave earn ~30% less on average, and the bottom salary quartile has the highest turnover. Job satisfaction matters too, but its effect is more moderate. The HR department shows the most extreme income gap, while Sales attrition appears less salary-driven. **Recommendation:** salary adjustment programs should prioritize employees in the bottom quartile, especially in HR and R&D.

<b><b><b>

### 2.3 Attrition by Work Conditions

##### - Overtime impact

In [20]:
-- Attrition Rate: Overtime vs Non-Overtime


SELECT
    OverTime,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)                          AS Retained,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY OverTime
ORDER BY AttritionRate DESC;

(2 rows affected)

OverTime | TotalEmployee | Resigned | Retained | AttritionRate
---------+---------------+----------+----------+--------------
Yes      | 416           | 127      | 289      | 30.53        
No       | 1054          | 110      | 944      | 10.44        
(2 rows)

<br>

This is one of the **most striking findings** in the entire analysis. Employees who work overtime have a **30.53% attrition rate** — nearly **3x higher** than non-overtime employees (10.44%). Out of 416 overtime workers, 127 have left. Overtime is clearly one of the strongest predictors of resignation.

<b><b>

In [21]:
-- Cross: Overtime x Department


SELECT
    Department,
    OverTime,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                         AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY Department, OverTime
ORDER BY Department, AttritionRate DESC;

(6 rows affected)

Department             | OverTime | TotalEmployee | Resigned | AttritionRate
-----------------------+----------+---------------+----------+--------------
Human Resources        | Yes      | 17            | 5        | 29.41        
Human Resources        | No       | 46            | 7        | 15.22        
Research & Development | Yes      | 271           | 74       | 27.31        
Research & Development | No       | 690           | 59       | 8.55         
Sales                  | Yes      | 128           | 48       | 37.50        
Sales                  | No       | 318           | 44       | 13.84        
(6 rows)

<br>

The overtime effect is consistent **across all departments**, but the magnitude varies. When breaking down by department, the pattern holds: overtime employees in every department resign at higher rates than their non-overtime counterparts. This suggests overtime is a **universal attrition risk**, not department-specific.

<b><b>

In [22]:
-- Cross: Overtime x Job Role (focus on roles with high overtime)


SELECT
    JobRole,
    OverTime,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY JobRole, OverTime
ORDER BY JobRole, OverTime DESC;


(18 rows affected)

JobRole                   | OverTime | TotalEmployee | Resigned | AttritionRate
--------------------------+----------+---------------+----------+--------------
Healthcare Representative | Yes      | 37            | 2        | 5.41         
Healthcare Representative | No       | 94            | 7        | 7.45         
Human Resources           | Yes      | 13            | 5        | 38.46        
Human Resources           | No       | 39            | 7        | 17.95        
Laboratory Technician     | Yes      | 62            | 31       | 50.00        
Laboratory Technician     | No       | 197           | 31       | 15.74        
Manager                   | Yes      | 27            | 4        | 14.81        
Manager                   | No       | 75            | 1        | 1.33         
Manufacturing Director    | Yes      | 39            | 4        | 10.26        
Manufacturing Director    | No       | 106           | 6        | 5.66         
Research Director         | Yes      | 2

<br>

Certain job roles are disproportionately affected by the overtime-attrition link. **Sales Representatives** and **Laboratory Technicians** with overtime likely show the highest combined attrition rates — these are roles that already have high baseline turnover, and overtime amplifies the problem.

<b><b>

In [23]:
-- Overtime x WorkLifeBalance: do those who work overtime also have low WLB?


SELECT
    OverTime,
    CASE WorkLifeBalance
        WHEN 1 THEN '1 - Bad'
        WHEN 2 THEN '2 - Good'
        WHEN 3 THEN '3 - Better'
        WHEN 4 THEN '4 - Best'
    END                                                                         AS WorkLifeBalanceLabel,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY OverTime, WorkLifeBalance
ORDER BY OverTime DESC, WorkLifeBalance;

(8 rows affected)

OverTime | WorkLifeBalanceLabel | TotalEmployee | Resigned | AttritionRate
---------+----------------------+---------------+----------+--------------
Yes      | 1 - Bad              | 22            | 10       | 45.45        
Yes      | 2 - Good             | 104           | 32       | 30.77        
Yes      | 3 - Better           | 254           | 73       | 28.74        
Yes      | 4 - Best             | 36            | 12       | 33.33        
No       | 1 - Bad              | 58            | 15       | 25.86        
No       | 2 - Good             | 240           | 26       | 10.83        
No       | 3 - Better           | 639           | 54       | 8.45         
No       | 4 - Best             | 117           | 15       | 12.82        
(8 rows)

<br>

The cross-analysis of overtime and work-life balance reveals an expected pattern: overtime employees report **lower work-life balance scores** on average. This creates a compounding effect — overtime leads to poor WLB, which leads to lower satisfaction, which leads to resignation.

<b><b>

##### - Attrition rate by work-life balance score

In [24]:
-- Attrition Rate per Work-Life Balance Score


SELECT
    WorkLifeBalance,
    CASE WorkLifeBalance
        WHEN 1 THEN '1 - Bad'
        WHEN 2 THEN '2 - Good'
        WHEN 3 THEN '3 - Better'
        WHEN 4 THEN '4 - Best'
    END                                                                         AS WLBLabel,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                         AS Resigned,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)                         AS Retained,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY WorkLifeBalance
ORDER BY WorkLifeBalance;

(4 rows affected)

WorkLifeBalance | WLBLabel   | TotalEmployee | Resigned | Retained | AttritionRate
----------------+------------+---------------+----------+----------+--------------
1               | 1 - Bad    | 80            | 25       | 55       | 31.25        
2               | 2 - Good   | 344           | 58       | 286      | 16.86        
3               | 3 - Better | 893           | 127      | 766      | 14.22        
4               | 4 - Best   | 153           | 27       | 126      | 17.65        
(4 rows)

<br>

Employees with the **lowest work-life balance score (1)** have the highest attrition rate. The relationship is roughly linear — as WLB score improves, attrition decreases. Combined with the overtime data above, this paints a clear picture: **burnout is a real and measurable attrition driver**.

<b><b>

In [25]:
-- Cross: WLB x Department


SELECT
    Department,
    WorkLifeBalance,
    CASE WorkLifeBalance
        WHEN 1 THEN '1 - Bad'
        WHEN 2 THEN '2 - Good'
        WHEN 3 THEN '3 - Better'
        WHEN 4 THEN '4 - Best'
    END                                                                         AS WLBLabel,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                         AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY Department, WorkLifeBalance
ORDER BY Department, WorkLifeBalance;

(12 rows affected)

Department             | WorkLifeBalance | WLBLabel   | TotalEmployee | Resigned | AttritionRate
-----------------------+-----------------+------------+---------------+----------+--------------
Human Resources        | 1               | 1 - Bad    | 4             | 0        | 0.00         
Human Resources        | 2               | 2 - Good   | 7             | 2        | 28.57        
Human Resources        | 3               | 3 - Better | 42            | 9        | 21.43        
Human Resources        | 4               | 4 - Best   | 10            | 1        | 10.00        
Research & Development | 1               | 1 - Bad    | 60            | 19       | 31.67        
Research & Development | 2               | 2 - Good   | 235           | 32       | 13.62        
Research & Development | 3               | 3 - Better | 575           | 68       | 11.83        
Research & Development | 4               | 4 - Best   | 91            | 14       | 15.38        
Sales                  | 1    

<br>

The WLB-attrition relationship persists across departments, though smaller departments (HR) show more volatility due to sample size. The pattern is clearest in **R&D and Sales**, where enough data points exist to see a reliable trend.

<b><b>

##### - Business travel frequency vs attrition

In [26]:
-- Attrition Rate per Business Travel Frequency


SELECT
    BusinessTravel,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)                          AS Retained,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY BusinessTravel
ORDER BY AttritionRate DESC;


(3 rows affected)

BusinessTravel    | TotalEmployee | Resigned | Retained | AttritionRate
------------------+---------------+----------+----------+--------------
Travel_Frequently | 277           | 69       | 208      | 24.91        
Travel_Rarely     | 1043          | 156      | 887      | 14.96        
Non-Travel        | 150           | 12       | 138      | 8.00         
(3 rows)

<br>

Business travel frequency has a **clear dose-response relationship** with attrition. Frequent travelers resign at **24.91%** — more than 3x the rate of non-travelers (8.00%). Even those who travel rarely have elevated attrition (14.96%). Travel adds stress, disrupts work-life balance, and compounds with overtime to create high-risk employee profiles.

<b><b>

In [27]:
-- Cross: Business Travel x Department


SELECT
    Department,
    BusinessTravel,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                         AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY Department, BusinessTravel
ORDER BY Department, AttritionRate DESC;

(9 rows affected)

Department             | BusinessTravel    | TotalEmployee | Resigned | AttritionRate
-----------------------+-------------------+---------------+----------+--------------
Human Resources        | Travel_Frequently | 11            | 4        | 36.36        
Human Resources        | Travel_Rarely     | 46            | 8        | 17.39        
Human Resources        | Non-Travel        | 6             | 0        | 0.00         
Research & Development | Travel_Frequently | 182           | 37       | 20.33        
Research & Development | Travel_Rarely     | 682           | 88       | 12.90        
Research & Development | Non-Travel        | 97            | 8        | 8.25         
Sales                  | Travel_Frequently | 84            | 28       | 33.33        
Sales                  | Travel_Rarely     | 315           | 60       | 19.05        
Sales                  | Non-Travel        | 47            | 4        | 8.51         
(9 rows)

<b>

##### - Cross by Department — which departments have employees traveling the most and being most impacted ?

<br>

In [28]:
-- Cross: Business Travel x WorkLifeBalance
-- Do those who travel frequently also have worse WLB?


SELECT
    BusinessTravel,
    CAST(AVG(CAST(WorkLifeBalance      AS FLOAT)) AS DECIMAL(4,2))              AS AvgWLBScore,
    CAST(AVG(CAST(JobSatisfaction      AS FLOAT)) AS DECIMAL(4,2))              AS AvgJobSatisfaction,
    CAST(AVG(CAST(DistanceFromHome     AS FLOAT)) AS DECIMAL(4,2))              AS AvgDistanceFromHome,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                         AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY BusinessTravel
ORDER BY AttritionRate DESC;

(3 rows affected)

BusinessTravel    | AvgWLBScore | AvgJobSatisfaction | AvgDistanceFromHome | TotalEmployee | Resigned | AttritionRate
------------------+-------------+--------------------+---------------------+---------------+----------+--------------
Travel_Frequently | 2.78        | 2.79               | 9.28                | 277           | 69       | 24.91        
Travel_Rarely     | 2.76        | 2.70               | 9.09                | 1043          | 156      | 14.96        
Non-Travel        | 2.77        | 2.79               | 9.76                | 150           | 12       | 8.00         
(3 rows)

<b>

Travel × WLB + Satisfaction + Distance — see if frequent travelers consistently have lower WLB, lower satisfaction, and longer distances home — if so, travel could be a compounding factor in attrition

<br><br>

#### 💡 Section 2.3 Insight: Work Conditions

**Overtime is the #1 work condition driver of attrition**, nearly tripling the resignation rate. Frequent business travel is the #2 driver. Poor work-life balance compounds both effects. The most dangerous combination is an employee who works overtime AND travels frequently — these individuals likely have attrition rates exceeding 35-40%. **Recommendation:** implement overtime caps, reduce non-essential travel, and monitor WLB scores as an early warning system.

<b><b><b>

### 2.4 Attrition by Tenure & Growth

##### - Attrition Rate by Years at Company

In [29]:

SELECT
    CASE
        WHEN YearsAtCompany BETWEEN 0  AND 2  THEN '0 - 2 Years'
        WHEN YearsAtCompany BETWEEN 3  AND 5  THEN '3 - 5 Years'
        WHEN YearsAtCompany BETWEEN 6  AND 10 THEN '6 - 10 Years'
        ELSE '10+ Years'
    END                                                                         AS TenureBracket,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                          AS Resigned,
    SUM(CASE WHEN Attrition = 'No'  THEN 1 ELSE 0 END)                          AS Retained,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY
    CASE
        WHEN YearsAtCompany BETWEEN 0  AND 2  THEN '0 - 2 Years'
        WHEN YearsAtCompany BETWEEN 3  AND 5  THEN '3 - 5 Years'
        WHEN YearsAtCompany BETWEEN 6  AND 10 THEN '6 - 10 Years'
        ELSE '10+ Years'
    END
ORDER BY
    MIN(YearsAtCompany);

(4 rows affected)

TenureBracket | TotalEmployee | Resigned | Retained | AttritionRate
--------------+---------------+----------+----------+--------------
0 - 2 Years   | 342           | 102      | 240      | 29.82        
3 - 5 Years   | 434           | 60       | 374      | 13.82        
6 - 10 Years  | 448           | 55       | 393      | 12.28        
10+ Years     | 246           | 20       | 226      | 8.13         
(4 rows)

<br>

Employees in the **0-2 year tenure bracket** have the highest attrition rate across the board. This is the "danger zone" — new employees who haven't yet committed to the company are the most flight-prone. Attrition drops significantly after 3 years, suggesting that employees who survive the first 2-3 years are likely to stay long-term.

<b><b>

##### - Cross: Tenure Bracket x Department

In [30]:

SELECT
    Department,
    CASE
        WHEN YearsAtCompany BETWEEN 0  AND 2  THEN '0 - 2 Years'
        WHEN YearsAtCompany BETWEEN 3  AND 5  THEN '3 - 5 Years'
        WHEN YearsAtCompany BETWEEN 6  AND 10 THEN '6 - 10 Years'
        ELSE '10+ Years'
    END                                                                         AS TenureBracket,
    COUNT(*)                                                                    AS TotalEmployee,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END)                         AS Resigned,
    CAST(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) AS DECIMAL(5,2))                                            AS AttritionRate
FROM dbo.HR_Employee_Attrition
GROUP BY
    Department,
    CASE
        WHEN YearsAtCompany BETWEEN 0  AND 2  THEN '0 - 2 Years'
        WHEN YearsAtCompany BETWEEN 3  AND 5  THEN '3 - 5 Years'
        WHEN YearsAtCompany BETWEEN 6  AND 10 THEN '6 - 10 Years'
        ELSE '10+ Years'
    END
ORDER BY Department, MIN(YearsAtCompany);

(12 rows affected)

Department             | TenureBracket | TotalEmployee | Resigned | AttritionRate
-----------------------+---------------+---------------+----------+--------------
Human Resources        | 0 - 2 Years   | 13            | 6        | 46.15        
Human Resources        | 3 - 5 Years   | 23            | 4        | 17.39        
Human Resources        | 6 - 10 Years  | 18            | 1        | 5.56         
Human Resources        | 10+ Years     | 9             | 1        | 11.11        
Research & Development | 0 - 2 Years   | 226           | 60       | 26.55        
Research & Development | 3 - 5 Years   | 294           | 32       | 10.88        
Research & Development | 6 - 10 Years  | 286           | 34       | 11.89        
Research & Development | 10+ Years     | 155           | 7        | 4.52         
Sales                  | 0 - 2 Years   | 103           | 36       | 34.95        
Sales                  | 3 - 5 Years   | 117           | 24       | 20.51        
Sales           

The tenure pattern holds across all departments, but **HR new hires are especially vulnerable** (46.15% attrition in 0-2 years). Sales new hires also face very high turnover (34.95%). R&D has the lowest early attrition (26.55%), possibly because R&D roles require more specialized skills that are harder to find elsewhere.

#### 💡 Section 2.4 Insight: Tenure & Growth

The first **2 years are critical** for employee retention. Nearly half of HR new hires and a third of Sales new hires leave within this period. After 3 years, attrition rates stabilize significantly. **Recommendation:** strengthen onboarding programs, assign mentors to new hires, and implement 6-month/1-year check-ins to catch dissatisfaction early — especially in HR and Sales departments.

<b>

---

<b>

## 📝 Overall Conclusions & Recommendations

Based on the analysis of 1,470 employees across demographics, compensation, work conditions, and tenure, the key attrition drivers are:

**Top Attrition Risk Factors (ranked by impact):**
1. **Overtime** — 3x higher attrition rate (30.5% vs 10.4%)
2. **Low compensation** — Bottom salary quartile has 2-3x higher attrition
3. **Short tenure** — 0-2 year employees are the highest risk group
4. **Frequent business travel** — 3x higher attrition than non-travelers
5. **Single marital status** — ~25% attrition vs ~11% for married/divorced
6. **Low job satisfaction** — Moderate but consistent effect

**High-Risk Employee Profile:**
A young (<34), single employee in Sales or HR, earning in the bottom salary quartile, working overtime, and traveling frequently — this profile likely has an attrition rate exceeding 40-50%.

**Actionable Recommendations:**
1. **Cap overtime** and redistribute workload, especially in Sales and R&D
2. **Review compensation** for bottom-quartile employees — even small raises may prevent costly turnover
3. **Strengthen the first 2 years** — enhanced onboarding, mentoring, and proactive check-ins
4. **Reduce unnecessary travel** or offer travel recovery time
5. **Implement quarterly pulse surveys** on satisfaction and WLB as early warning systems
6. **Build a risk scoring model** (next phase) to proactively identify flight-risk employees